In [1]:
%load_ext autoreload
%autoreload 2

This notebook demonstrates how to train a language model to play **Wordle** using
**Group Relative Policy Optimization (GRPO)** with TRL and the clemcore OpenEnv integration.

You will learn how to:

1. Set up the environment and install dependencies.
2. Connect to a Wordle game server using OpenEnv.
3. Implement a GRPO-compatible agent that tracks episode trajectories.
4. Run GRPO training with TRL using the OpenEnv game as the environment.

**Key concepts:**
- **GRPO**: A reinforcement learning algorithm that compares K completions from the same prompt to compute advantages, without requiring a separate reference model.
- **OpenEnv**: A Gymnasium-style API for interacting with text-based environments.
- **rollout_func**: A custom function that lets TRL collect trajectories from an environment instead of using standard text generation.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/phisad/playpen/blob/main/examples/openenv/wordle-trl.ipynb
)

# 1. Environment setup and dependency installation

## 1.1. Setup environment

We start by specifying:

- The game name (here we use `"wordle"`, but this pattern works for any 1‑player game).
- `CLEMBENCH_HOME`, the local path where the `clembench` repository is located.
  This environment variable is used by the CLI and Python APIs to locate games.

In [2]:
import os

# Specify the game name here (this code can be adapted to any 1-player game)
GAME_NAME = "wordle"

# Local clone location of the clembench repository
CLEMBENCH_HOME = os.path.expanduser("~/project/sadler/clembench")

# Expose CLEMBENCH_HOME so the clem framework can find the games
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

In [3]:
# Sanity check: version + confirm that the game is an available game
!clem --version
!clem list games -s $GAME_NAME

/bin/bash: line 1: clem: command not found
/bin/bash: line 1: clem: command not found


# 2. Connect to the Wordle game server

The Wordle game runs as an OpenEnv server that our agent will interact with.
Each call to `env.reset()` starts a new game with a random target word, and
`env.step(action)` submits a guess and returns feedback.

## 2.1. Starting the server and connecting the client

Open a terminal in the notebook folder and run the `clem serve` command to start the OpenEnv environment:

```bash
clem serve --game wordle --port 9000 --split train
```

The output should look similar to:
```
INFO:     Started server process [73210]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
```

From this log you obtain the host and port (here `http://0.0.0.0:8000`) that the client should connect to.

In [4]:
from clemcore.clemgame import ClemGameEnv

game_env = ClemGameEnv(base_url="http://0.0.0.0:9000")

# 3. Implement the GRPO-compatible Wordle agent

To train with GRPO using an interactive environment, we need to:

1. **Track the full episode trajectory** as a single prompt-completion pair
2. **Distinguish model tokens from environment tokens** using an `env_mask`
3. **Capture log probabilities** for each generated token

The trajectory structure follows TRL's pattern:
- `prompt_ids`: The initial game prompt (set once at episode start)
- `completion_ids`: All model outputs + environment feedback concatenated across turns
- `logprobs`: Log probability for each token (0.0 for environment tokens)
- `env_mask`: 1 for model-generated tokens, 0 for environment tokens

GRPO only computes gradients on tokens where `env_mask=1`, but uses the full
trajectory to calculate rewards and advantages.

## 3.1. Define data containers for episode trajectories

We define two dataclasses:
- `GrpoEpisodeRollout`: Holds a single episode's trajectory data
- `GrpoEpisodeRollouts`: Aggregates multiple episodes for a training batch

These will be converted to a dictionary via `asdict()` and returned to TRL's
`GRPOTrainer`, which expects keys: `prompt_ids`, `completion_ids`, `logprobs`, and `env_mask`.

In [5]:
from dataclasses import dataclass, field
from playpen.agents.openenv import ClemGameEnvAgent


@dataclass
class GrpoEpisodeRollout:
    """Collect training info about a single episode.
    
    Following TRL's pattern, we accumulate all turns into a single completion
    sequence, using env_masks to distinguish model tokens (1) from env tokens (0).
    """
    prompt_ids: list[int] = field(default_factory=list)
    completion_ids: list[int] = field(default_factory=list)
    logprobs: list[float] = field(default_factory=list)
    env_masks: list[int] = field(default_factory=list)
    reward: float = 0.0  # Terminal reward from the environment

    def reset(self):
        self.prompt_ids.clear()
        self.completion_ids.clear()
        self.logprobs.clear()
        self.env_masks.clear()
        self.reward = 0.0


@dataclass
class GrpoEpisodeRollouts:
    """Collect training info about all episodes for a batch."""
    prompt_ids: list[list[int]] = field(default_factory=list)
    completion_ids: list[list[int]] = field(default_factory=list)
    logprobs: list[list[float]] = field(default_factory=list)
    env_mask: list[list[int]] = field(default_factory=list)
    rewards: list[float] = field(default_factory=list)  # Rewards for each episode

    def append(self, rollout: GrpoEpisodeRollout):
        """Append a completed episode rollout."""
        self.prompt_ids.append(list(rollout.prompt_ids))
        self.completion_ids.append(list(rollout.completion_ids))
        self.logprobs.append(list(rollout.logprobs))
        self.env_mask.append(list(rollout.env_masks))
        self.rewards.append(rollout.reward)

    def reset(self):
        self.prompt_ids.clear()
        self.completion_ids.clear()
        self.logprobs.clear()
        self.env_mask.clear()
        self.rewards.clear()


.--------------..--------------..--------------..--------------..--------------..--------------..--------------.
|   ______     ||   _____      ||      __      ||  ____  ____  ||   ______     ||  _________   || ____  _____  |
|  |_   __ \   ||  |_   _|     ||     /  \     || |_  _||_  _| ||  |_   __ \   || |_   ___  |  |||_   \|_   _| |
|    | |__) |  ||    | |       ||    / /\ \    ||   \ \  / /   ||    | |__) |  ||   | |_  \_|  ||  |   \ | |   |
|    |  ___/   ||    | |   _   ||   / ____ \   ||    \ \/ /    ||    |  ___/   ||   |  _|  _   ||  | |\ \| |   |
|   _| |_      ||   _| |__/ |  || _/ /    \ \_ ||    _|  |_    ||   _| |_      ||  _| |___/ |  || _| |_\   |_  |
|  |_____|     ||  |________|  |||____|  |____|||   |______|   ||  |_____|     || |_________|  |||_____|\____| |
'--------------''--------------''--------------''--------------''--------------''--------------''--------------'



/project/sadler/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3.2. Implement the WordleAgent

The `WordleAgent` extends `ClemAgent` and handles:

1. **Generation**: Uses TRL's `generate_rollout_completions()` with vLLM for fast inference
2. **Trajectory tracking**: Accumulates tokens and logprobs across all turns

Key methods:
- `track_env_completion()`: On first turn, captures the initial prompt. On subsequent turns,
  tokenizes the environment feedback and adds it with `env_mask=0`.
- `track_agent_completion()`: Adds model-generated tokens with `env_mask=1`.
- `act()`: Orchestrates tracking → generation → tracking → return response.

The agent uses `self.history` (inherited from `ClemAgent`) to build the full
conversation context for each generation call.

In [6]:
import trl
from playpen.agents import ClemObservation, ClemAgent
from trl.experimental.openenv import generate_rollout_completions


class WordleAgent(ClemAgent):
    """Agent that plays Wordle using TRL's GRPOTrainer for generation.
    
    Handles all tokenization and env_mask logic internally, accumulating
    the full episode trajectory for GRPO training.

    This implementation is based on the wordle openenv example given in the trl repository at
    https://github.com/huggingface/trl/blob/v0.28.0/examples/scripts/openenv/wordle.py
    """

    def __init__(self, trainer: trl.GRPOTrainer):
        super().__init__()
        self.trainer = trainer
        self.tokenizer = trainer.processing_class
        self.episode = GrpoEpisodeRollout()
        self._first_turn = True

    def track_env_completion(self, last: ClemObservation):
        if self._first_turn:  # On the first turn, set prompt_ids from the initial observation
            prompt_text = self.tokenizer.apply_chat_template(self.history, add_generation_prompt=True, tokenize=False)
            self.episode.prompt_ids = self.tokenizer.encode(prompt_text, add_special_tokens=False)
            self._first_turn = False
            return
        # Not first turn: the observation content is env feedback from the previous action
        env_feedback_ids = self.tokenizer.encode(last.content, add_special_tokens=False)
        self.episode.completion_ids.extend(env_feedback_ids)
        self.episode.logprobs.extend([0.0] * len(env_feedback_ids))
        self.episode.env_masks.extend([0] * len(env_feedback_ids))

    def track_agent_completion(self, outputs):
        self.episode.completion_ids.extend(outputs["completion_ids"])
        self.episode.logprobs.extend(outputs["logprobs"])
        self.episode.env_masks.extend([1] * len(outputs["completion_ids"]))

    def act(self, last: ClemObservation) -> str:
        self.track_env_completion(last)
        # We use text here so that we can better track the agent tokens versus the env tokens
        prompt_text = self.tokenizer.apply_chat_template(self.history, add_generation_prompt=True, tokenize=False)
        outputs = generate_rollout_completions(self.trainer, [prompt_text])[0]
        self.track_agent_completion(outputs)
        response = outputs.get("text") or self.tokenizer.decode(outputs["completion_ids"], skip_special_tokens=True)
        return response

    def get_episode(self) -> "GrpoEpisodeRollout":
        """Return the accumulated episode rollout for GRPO training."""
        return self.episode

    def reset(self):
        """Reset agent state for a new episode."""
        super().reset()
        self.episode.reset()
        self._first_turn = True

/tmp/ipykernel_353883/2011033569.py:3: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.openenv import generate_rollout_completions


## 3.3. Define the rollout function for TRL

TRL's `GRPOTrainer` supports a `rollout_func` parameter that overrides the default
text generation loop. This lets us interact with the OpenEnv game instead.

> **Important:** TRL only supports `rollout_func` when `use_vllm=True`. The vLLM
> backend provides the `generate_rollout_completions()` helper that returns token IDs
> and log probabilities in the format GRPO expects. Without vLLM, you would need to
> implement your own generation with logprob tracking.

**How it works:**
1. TRL calls `rollout_func(prompts, trainer)` with prompts repeated K times (K = `num_generations`)
2. We run K episodes sequentially (one per prompt), collecting trajectories
3. We return a dict with `prompt_ids`, `completion_ids`, `logprobs`, and `env_mask`
4. TRL uses this data for advantage computation and policy gradient updates

Note: Since we only have one game server, episodes run sequentially. For better
throughput, you could spawn multiple game servers and parallelize.

In [ ]:
from dataclasses import asdict
import math
import random

def rollout_reward(history: list[dict], reward: float):
    """Apply a reward shaping hack for the degenerate all-abort case.
    
    Terminal rewards from the environment:
        - win:   +1.0
        - loss:   0.0
        - abort: -1.0  (model output violated the expected format)

    GRPO computes advantages by normalizing rewards within each group of K
    completions: ``advantage = (r - mean(r)) / (std(r) + eps)``.
    If **every** episode in a group aborts, all rewards are identical (-1.0),
    so the numerator ``r - mean(r) = -1.0 - (-1.0) = 0`` for every episode,
    every advantage = 0, and the gradient is exactly zero —
    the model receives no learning signal at all.

    To break this symmetry we replace each abort reward with a small uniformly
    sampled negative value in (-1, 0).  This introduces variance within an
    all-abort group so that ``r - mean(r) ≠ 0`` and the loss becomes non-zero,
    giving the model at least a weak signal to escape the degenerate regime.

    Best practice: run SFT first so the model already produces valid format
    output before starting GRPO, which avoids this situation entirely.
    """
    if math.isclose(reward, -1.0, abs_tol=1e-6):
        reward = -random.random()
    return reward

def rollout_episode(env: ClemGameEnv, agent: ClemGameEnvAgent) -> GrpoEpisodeRollout:
    """Play a single episode of Wordle and collect GRPO training data.
    
    The agent handles all tokenization and env_mask logic internally.
    
    Returns:
        GrpoEpisodeRollout with accumulated prompt_ids, completion_ids, logprobs, env_masks, and reward.
    """
    obs = env.reset()
    while not obs.done:
        action = agent(obs)
        obs = env.step(action)
    rollout = agent.wrapped_agent.get_episode()
    rollout.reward = rollout_reward(agent.wrapped_agent.history, obs.reward)
    return rollout


def rollout_func(prompts: list[str], trainer: trl.GRPOTrainer) -> dict:
    """Custom rollout function for TRL GRPOTrainer with OpenEnv.
    
    Note:
        rollout_func receives prompts already duplicated K times (num_generations).
        For example, with batch size 4 and num_generations 8, there are 32 prompts.
        Since Wordle always starts with the same initial state, we ignore the prompt
        content and just run that many episodes.
    """
    agent = ClemGameEnvAgent(WordleAgent(trainer))
    rollouts = GrpoEpisodeRollouts()
    try:
        for _ in prompts:
            rollout = rollout_episode(game_env, agent)
            rollouts.append(rollout)
            agent.reset()
        return asdict(rollouts)
    finally:
        rollouts.reset()
        agent.reset()


def reward_env(completions: list[str], **kwargs) -> list[float]:
    """Extract environment rewards passed from rollout_func.
    
    TRL calls this function with completions and any extra fields from the rollout dict.
    The 'rewards' field contains the terminal reward for each episode (1.0 = win, 0.0 = loss, -1.0 = abort).
    """
    return kwargs.get("rewards", [0.0] * len(completions))

# 4. Configure and run GRPO training

Now we set up the `GRPOTrainer` with:
- **vLLM colocate mode**: Fast inference on the same GPU as training
- **LoRA**: Only trains adapter weights, not the full model
- **8-bit quantization** (optional): Further reduces memory for smaller GPUs

**Memory estimates for Llama 3 8B + LoRA:**

| Configuration | GPU Memory |
|---------------|------------|
| bf16 (no quantization) | ~45 GB |
| 8-bit quantization | ~28 GB |

On an A100 80GB, you can skip quantization. Use 8-bit for 40GB A100 or 24GB consumer GPUs.

## 4.1. Load the training dataset

We load game instances from the playpen-data dataset. Each instance represents
a game configuration (e.g., a specific target word). The dataset is filtered
to only include Wordle instances.

Note: In this setup, the dataset mainly controls how many training steps we run.
The actual game content comes from the OpenEnv server, which generates new games
on each `env.reset()` call.

**Dummy `prompt` column:** `GRPOTrainer` requires the dataset to have a `"prompt"`
column, because it uses this field to build the initial input for the model.
In our case the real prompt comes from the game environment (via `env.reset()`),
so the dataset's prompt value is never actually used for generation.
We still have to add the column to satisfy TRL's schema validation — hence the
`"dummy"` placeholder values.

In [8]:
from datasets import load_dataset

dataset = load_dataset("colab-potsdam/playpen-data", "instances", split="train")
dataset = dataset.filter(lambda game_instance: game_instance["game"] == "wordle")
dataset = dataset.add_column("prompt", ["dummy"] * len(dataset)) # GRPOTrainer expects a prompt field (which is never used in our case)

## 4.2. Configure and start training

Key configuration options:

**GRPOConfig:**
- `use_vllm=True, vllm_mode="colocate"`: Use vLLM for generation on the same GPU
- `num_generations=4`: Compare 4 completions per prompt for advantage estimation
- `disable_dropout=True`: Ensures consistent policy during generation and training
- `max_completion_length=2048`: Must fit the full episode (6 turns × explanation + guess)

**LoraConfig:**
- `r=16, lora_alpha=32`: LoRA rank and scaling factor
- `target_modules="all-linear"`: Apply LoRA to all linear layers

**BitsAndBytesConfig (optional):**
- `load_in_8bit=True`: Quantize base model to reduce memory
- Set to `None` on 80GB+ GPUs where memory is not a constraint

**Memory estimates for Llama 3 8B + LoRA:**

| Configuration | GPU Memory |
|---------------|------------|
| bf16 (no quantization) | ~45 GB |
| 8-bit quantization | ~28 GB |

On an A100 80GB, you can skip quantization. Use 8-bit for 40GB A100 or 24GB consumer GPUs.

**Why `vllm_gpu_memory_utilization=0.45`:**
In colocate mode the model is loaded **twice** into GPU memory:
once by Transformers (for the training / gradient-update side) and once by vLLM
(for fast rollout generation). Both copies live on the same GPU simultaneously,
so each must fit in roughly half the available VRAM. Setting `vllm_gpu_memory_utilization=0.45`
reserves ≤ 45 % of GPU memory for vLLM, leaving the other ~55 % for the
Transformers copy plus optimizer states and activations.
Raise this value if you have spare headroom; lower it (or switch to 8-bit
quantization) if you see out-of-memory errors.

> **Troubleshooting:** If training hangs after a few iterations with `vllm_mode="colocate"` + LoRA,
> this is a [known issue](https://github.com/huggingface/trl/issues/3671). Use server mode with 2 GPUs instead:
> ```bash
> # In a separate terminal: Start vLLM server on GPU 0
> CUDA_VISIBLE_DEVICES=0 trl vllm-serve --model meta-llama/Meta-Llama-3-8B-Instruct
> ```
> Then set `vllm_mode="server"` and run the notebook with `CUDA_VISIBLE_DEVICES=1`.

> **Note on evaluation:** TRL's built-in evaluation (`do_eval=True`) does not support `rollout_func`.
> Evaluation uses standard text generation, not interactive environments. To evaluate the trained
> model on Wordle, run `playpen eval <model-name> -g wordle` after training completes.

The training loop will:
1. Sample a batch of prompts from the dataset
2. Call `rollout_func` to play episodes and collect trajectories
3. Compute advantages by comparing rewards across the K completions
4. Update LoRA weights via policy gradient

In [9]:
from peft import LoraConfig
from datetime import datetime

TIMESTAMP = datetime.now().strftime("%b%d_%H-%M")

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
#MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct" # for debugging
grpo_config = trl.GRPOConfig(
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.45,
    # vllm_gpu_memory_utilization=0.05, # for smol
    num_generations=2,  # Default
    per_device_train_batch_size=2,  # Default
    num_train_epochs=3,
    disable_dropout=True,
    max_completion_length=2048,  # Should capture full episode; note that the model is asked to give an explanation
    output_dir=f"models/grpo/wordle/{MODEL_ID}",
    report_to="tensorboard",
    logging_dir=f"/cache/tensorboard-logdir/grpo/wordle/{MODEL_ID}/{TIMESTAMP}",
    log_completions=True
)
peft_config = LoraConfig(  # see https://huggingface.co/docs/trl/sft_trainer#training-adapters
    r=8, lora_alpha=16,
    lora_dropout=0.05,
    target_modules="all-linear",
    modules_to_save=["lm_head", "embed_token"],
    task_type="CAUSAL_LM",
)

# Optional: 8-bit quantization to reduce memory (~28GB vs ~45GB for bf16)
bnb_config = None  # Default: Set to None on 80GB+ GPUs

# Uncomment for 40GB A100 or consumer GPUs
# from transformers import BitsAndBytesConfig
# bnb_config = BitsAndBytesConfig(load_in_8bit=True)

grpo_trainer = trl.GRPOTrainer(
    model=MODEL_ID,
    rollout_func=rollout_func,  # repeats each game instance in batch K times (using RepeatSampler)
    reward_funcs=reward_env,  # extracts terminal rewards from episode rollouts
    train_dataset=dataset,
    args=grpo_config,
    peft_config=peft_config
)

# Train on the dataset; this will save only the adapters to the checkpoints directory
grpo_trainer.train()

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.28it/s]
/project/sadler/venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
/tmp/ipykernel_353883/560988964.py:41: UserWarning: You are importing from 'rollout_func', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  grpo_trainer = trl.GRPOTrainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
2026-02-19 13:59:21,035	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run 

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:02,  1.40it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:01<00:01,  1.28it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:02<00:00,  1.24it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.64it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.49it/s]

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 3/3 [00:00<00:00,  3.81it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 2/2 [00:00<00:00,  5.41it/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.
Could not estimate the number of tokens of the input, floating

Step,Training Loss
10,0.000000
20,-0.000000
30,0.000800
40,0.000900
50,0.000600
60,-0.000000
70,0.000000


╭──────────────────────────────────────────────────── Step 10 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                              ┃ reward_env ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                     │ explanation: this is a common           │      -0.28 │      0.71 │ │
│ │                                          │ five-letter English word, and I am      │            │           │ │
│ │ You are a language wizard who likes to   │ starting my guess with a word that is   │            │           │ │
│ │ guess words by using the given rules.    │ likely to contain a common letter       │            │           │ │
│ │                                          │ combination and has no consecutive      │            │           │ │
│ │ Welcome to Wordle! You have six attempts │ repeated letters.                       │            │           │ │
│ │ to guess the target word, a valid        │ guess: houseguess_feedback: h<red>      │            │           │ │
│ │ English word of five lowercase letters   │ o<red> u<red> s<red> e<yellow>          │            │           │ │
│ │ (a-z). Please use the tags               │                                         │            │           │ │
│ │ "explanation:" and "guess:" to provide a │ Provide your response only in this      │            │           │ │
│ │ concise explanation for each guess.      │ format.                                 │            │           │ │
│ │                                          │ explanation: details                    │            │           │ │
│ │ For instance, if your guess is "apple",  │ guess: wordexplanation: the first five  │            │           │ │
│ │ your response should be                  │ letters are not in the correct word,    │            │           │ │
│ │ explanation: this is a common            │ and only the last letter "e" is         │            │           │ │
│ │ five-letter English word, and I am       │ partially correct, which is a common    │            │           │ │
│ │ starting my guess with this word.        │ occurrence in many words. I will try to │            │           │ │
│ │ guess: apple                             │ reuse the correct letter "e" in a       │            │           │ │
│ │                                          │ different position.                     │            │           │ │
│ │ After each guess, your answer will be    │ guess: themeguess_feedback: t<red>      │            │           │ │
│ │ validated, and you will receive feedback │ h<red> e<yellow> m<red> e<red>          │            │           │ │
│ │ indicating which letters are correct     │                                         │            │           │ │
│ │ (green), which letters are correct but   │ Provide your response only in this      │            │           │ │
│ │ in the wrong position (yellow), and      │ format.                                 │            │           │ │
│ │ which letters are incorrect (red). This  │ explanation: details                    │            │           │ │
│ │ feedback can be useful in determining    │ guess: wordexplanation: Since the       │            │           │ │
│ │ which letters to include or exclude in   │ letter "e" was already guessed          │            │           │ │
│ │ your next guess.                         │ correctly in the previous attempt, I    │            │           │ │
│ │                                          │ will build upon that by keeping it in   │            │           │ │
│ │ For example, the feedback for "apple"    │ the same position in my next guess. The │            │           │ │
│ │ might be:                                │ letters "

...

╭──────────────────────────────────────────────────── Step 60 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                              ┃ reward_env ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                     │ explanation: I'm going to start with a  │      -0.45 │      0.71 │ │
│ │                                          │ common five-letter English word that is │            │           │ │
│ │ You are a language wizard who likes to   │ often used in everyday conversation.    │            │           │ │
│ │ guess words by using the given rules.    │ guess: houseguess_feedback: h<red>      │            │           │ │
│ │                                          │ o<yellow> u<red> s<red> e<green>        │            │           │ │
│ │ Welcome to Wordle! You have six attempts │                                         │            │           │ │
│ │ to guess the target word, a valid        │ Provide your response only in this      │            │           │ │
│ │ English word of five lowercase letters   │ format.                                 │            │           │ │
│ │ (a-z). Please use the tags               │ explanation: details                    │            │           │ │
│ │ "explanation:" and "guess:" to provide a │ guess: wordexplanation: This guess      │            │           │ │
│ │ concise explanation for each guess.      │ feedback shows that the letter "e" is   │            │           │ │
│ │                                          │ present in the target word, but not in  │            │           │ │
│ │ For instance, if your guess is "apple",  │ the correct position. The letters "h",  │            │           │ │
│ │ your response should be                  │ "u", and "s" are not present in the     │            │           │ │
│ │ explanation: this is a common            │ target word. The letter "o" is present  │            │           │ │
│ │ five-letter English word, and I am       │ in the target word, but not in the      │            │           │ │
│ │ starting my guess with this word.        │ correct position. I'll try to use this  │            │           │ │
│ │ guess: apple                             │ information to arrive at a new guess.   │            │           │ │
│ │                                          │ guess: tone                             │            │           │ │
│ │ After each guess, your answer will be    │                                         │            │           │ │
│ │ validated, and you will receive feedback │                                         │            │           │ │
│ │ indicating which letters are correct     │                                         │            │           │ │
│ │ (green), which letters are correct but   │                                         │            │           │ │
│ │ in the wrong position (yellow), and      │                                         │            │           │ │
│ │ which letters are incorrect (red). This  │                                         │            │           │ │
│ │ feedback can be useful in determining    │                                         │            │           │ │
│ │ which letters to include or exclude in   │                                         │            │           │ │
│ │ your next guess.                         │                                         │            │           │ │
│ │                                          │                                         │            │           │ │
│ │ For example, the feedback for "apple"    │                                         │            │           │ │
│ │ might be:                                │          

╭──────────────────────────────────────────────────── Step 70 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                              ┃ reward_env ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                     │ explanation: this is a common           │      -0.22 │      0.71 │ │
│ │                                          │ five-letter English word, and I am      │            │           │ │
│ │ You are a language wizard who likes to   │ starting my guess with a basic random   │            │           │ │
│ │ guess words by using the given rules.    │ word.                                   │            │           │ │
│ │                                          │ guess: horseguess_feedback: h<red>      │            │           │ │
│ │ Welcome to Wordle! You have six attempts │ o<red> r<yellow> s<red> e<yellow>       │            │           │ │
│ │ to guess the target word, a valid        │                                         │            │           │ │
│ │ English word of five lowercase letters   │ Provide your response only in this      │            │           │ │
│ │ (a-z). Please use the tags               │ format.                                 │            │           │ │
│ │ "explanation:" and "guess:" to provide a │ explanation: details                    │            │           │ │
│ │ concise explanation for each guess.      │ guess: wordexplanation: Since the       │            │           │ │
│ │                                          │ letters "r" and "e" appeared in the     │            │           │ │
│ │ For instance, if your guess is "apple",  │ feedback as yellow, I will focus on     │            │           │ │
│ │ your response should be                  │ words that contain these letters and    │            │           │ │
│ │ explanation: this is a common            │ try to build upon them. I'll also avoid │            │           │ │
│ │ five-letter English word, and I am       │ using the letters "h" and "s" since     │            │           │ │
│ │ starting my guess with this word.        │ they were marked as red.                │            │           │ │
│ │ guess: apple                             │ guess: rise                             │            │           │ │
│ │                                          │                                         │            │           │ │
│ │ After each guess, your answer will be    │                                         │            │           │ │
│ │ validated, and you will receive feedback │                                         │            │           │ │
│ │ indicating which letters are correct     │                                         │            │           │ │
│ │ (green), which letters are correct but   │                                         │            │           │ │
│ │ in the wrong position (yellow), and      │                                         │            │           │ │
│ │ which letters are incorrect (red). This  │                                         │            │           │ │
│ │ feedback can be useful in determining    │                                         │            │           │ │
│ │ which letters to include or exclude in   │                                         │            │           │ │
│ │ your next guess.                         │                                         │            │           │ │
│ │                                          │                                         │            │           │ │
│ │ For example, the feedback for "apple"    │                                         │            │           │ │
│ │ might be:                                │          

/project/sadler/venv/lib/python3.10/site-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69971956-65814f5a7ca89aea32495736;47ace4d4-74bb-4e65-b452-84017375f2fe)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3-8B-Instruct.
  warnings.warn(
/project/sadler/venv/lib/python3.10/site-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in meta-llama/Meta-Llama-3-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


╭──────────────────────────────────────────────────── Step 72 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                              ┃ reward_env ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                     │ explanation: this is a five-letter      │      -0.10 │      0.70 │ │
│ │                                          │ English word and I am starting my guess │            │           │ │
│ │ You are a language wizard who likes to   │ from a common letter combination.       │            │           │ │
│ │ guess words by using the given rules.    │ guess: houseguess_feedback: h<red>      │            │           │ │
│ │                                          │ o<yellow> u<red> s<red> e<green>        │            │           │ │
│ │ Welcome to Wordle! You have six attempts │                                         │            │           │ │
│ │ to guess the target word, a valid        │ Provide your response only in this      │            │           │ │
│ │ English word of five lowercase letters   │ format.                                 │            │           │ │
│ │ (a-z). Please use the tags               │ explanation: details                    │            │           │ │
│ │ "explanation:" and "guess:" to provide a │ guess: wordexplanation: This feedback   │            │           │ │
│ │ concise explanation for each guess.      │ suggests that the word contains the     │            │           │ │
│ │                                          │ letter "e" in the correct position,     │            │           │ │
│ │ For instance, if your guess is "apple",  │ which is a good starting point. It also │            │           │ │
│ │ your response should be                  │ indicates that the letters "o" and "s"  │            │           │ │
│ │ explanation: this is a common            │ are present in the word, but not in the │            │           │ │
│ │ five-letter English word, and I am       │ correct position. The letters "h" and   │            │           │ │
│ │ starting my guess with this word.        │ "u" are not present in the word. I'll   │            │           │ │
│ │ guess: apple                             │ try to incorporate this information     │            │           │ │
│ │                                          │ into my next guess.                     │            │           │ │
│ │ After each guess, your answer will be    │ guess: houseguess_feedback: h<red>      │            │           │ │
│ │ validated, and you will receive feedback │ o<yellow> u<red> s<red> e<green>        │            │           │ │
│ │ indicating which letters are correct     │                                         │            │           │ │
│ │ (green), which letters are correct but   │ Provide your response only in this      │            │           │ │
│ │ in the wrong position (yellow), and      │ format.                                 │            │           │ │
│ │ which letters are incorrect (red). This  │ explanation: details                    │            │           │ │
│ │ feedback can be useful in determining    │ guess: wordexplanation: Since the       │            │           │ │
│ │ which letters to include or exclude in   │ letter 'e' is green, I'm going to focus │            │           │ │
│ │ your next guess.                         │ on it and make a guess that             │            │           │ │
│ │                                          │ incorporates it. The letters 'o' and    │            │           │ │
│ │ For example, the feedback for "apple"    │ 'h' were also somewhat correct, so I'll │            │           │ │
│ │ might be:                                │ try to in

TrainOutput(global_step=72, training_loss=0.000327543937601215, metrics={'train_runtime': 532.0754, 'train_samples_per_second': 0.135, 'train_steps_per_second': 0.135, 'total_flos': 0.0, 'train_loss': 0.000327543937601215})